In [1]:
# Day 3 introduces LangChain, prompt templates, chains, conversation memory, and testing. 
# Today your classifier will become a LangChain-based complaint assistant.
%pip install -qU langchain langchain-groq

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from getpass import getpass
from langchain_groq import ChatGroq
os.environ["GROQ_API_KEY"] = getpass("Groq API Key: ")
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

Groq API Key:  ········


In [3]:
SYSTEM = """
You are an AI customer complaint assistant.
Classify complaints into Payment, Delivery, Product,
Refund, or Other.
Identify priority: Low, Medium, or High.
Include reason, sentiment, and recommended action.
Respond clearly using JSON.
Do not invent information.
"""

In [4]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])
chain = prompt | llm | StrOutputParser()
history = []

In [5]:
def ask(question):
    answer = chain.invoke({
        "history": history,
        "question": question
    })
    history.extend([
        ("human", question),
        ("ai", answer)
    ])
    return answer
print(ask("My payment was deducted twice."))
print(ask("What category was my previous complaint?"))

```json
{
  "category": "Payment",
  "priority": "High",
  "reason": "Duplicate deduction of the same payment",
  "sentiment": "Negative",
  "action": "Investigate the transaction history, issue a refund for the duplicate charge, and confirm the resolution with the customer."
}
```
```json
{
  "category": "Payment"
}
```


In [7]:
import csv
tests = [
    ["My payment was deducted twice.", "Payment"],
    ["My delivery is three days late.", "Delivery"],
    ["The product I received is damaged.", "Product"],
    ["I requested a refund but haven't received it.", "Refund"],
    ["I received the wrong product.", "Product"],
    ["I was charged but my order wasn't placed.", "Payment"],
    ["My tracking hasn't updated.", "Delivery"],
    ["My product stopped working.", "Product"],
    ["I want to cancel and get my money back.", "Refund"],
    ["I cannot log into my account.", "Other"]
]
with open("complaint_tests.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["question", "must_include"])
    writer.writerows(tests)
print("Test dataset created!")

Test dataset created!


In [8]:
def run_tests(path="complaint_tests.csv"):
    with open(path, newline="") as f:
        rows = list(csv.DictReader(f))
    passed = 0
    for row in rows:
        answer = chain.invoke({
            "history": [],
            "question": row["question"]
        })
        ok = row["must_include"].lower() in answer.lower()
        passed += ok
        print("PASS" if ok else "FAIL", "|", row["question"])
    print(f"\n{passed}/{len(rows)} passed ({passed/len(rows):.0%})")
run_tests()

PASS | My payment was deducted twice.
PASS | My delivery is three days late.
PASS | The product I received is damaged.
PASS | I requested a refund but haven't received it.
PASS | I received the wrong product.
PASS | I was charged but my order wasn't placed.
PASS | My tracking hasn't updated.
PASS | My product stopped working.
PASS | I want to cancel and get my money back.
PASS | I cannot log into my account.

10/10 passed (100%)


In [9]:
from langchain.agents import create_agent
agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=SYSTEM
)
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "My refund has not arrived."}
    ]
})
print(result["messages"][-1].content)

```json
{
  "category": "Refund",
  "priority": "High",
  "reason": "Refund has not been received by the customer.",
  "sentiment": "Negative",
  "recommended_action": "Check the refund status in the order history, verify the refund amount and date, and contact customer support with the order number and refund reference to expedite resolution."
}
```


In [10]:
llm_creative = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=1.0
)
creative_chain = prompt | llm_creative | StrOutputParser()
question = "My package arrived late. Classify this complaint."
print(creative_chain.invoke({
    "history": [],
    "question": question
}))

```json
{
  "complaintType": "Delivery",
  "priority": "Medium",
  "reason": "Package arrived later than the promised delivery date.",
  "sentiment": "Negative",
  "recommendedAction": "Apologize for the delay, provide the updated tracking information, and offer a small discount or expedited shipping on the next order as a goodwill gesture."
}
```


In [13]:
def get_category_info(category: str) -> str:
    """Return information about a complaint category."""
    return categories.get(category, "Unknown category")
def suggest_priority(category: str) -> str:
    """Suggest a default priority for a complaint category."""
    priorities = {
        "Payment": "High",
        "Delivery": "Medium",
        "Product": "High",
        "Refund": "Medium",
        "Other": "Low"
    }
    return priorities.get(category, "Unknown category")

In [15]:
categories = {
    "Payment": "Payment or transaction problems",
    "Delivery": "Shipping or tracking problems",
    "Product": "Damaged, defective, or wrong items",
    "Refund": "Refund or cancellation issues",
    "Other": "Other customer support problems"
}
print(get_category_info("Payment"))
print(suggest_priority("Delivery"))
print(suggest_priority("Product"))

Payment or transaction problems
Medium
High


In [16]:
llm_creative = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=1.0
)
creative_chain = prompt | llm_creative | StrOutputParser()
question = "My package arrived late. Classify this complaint."
result = creative_chain.invoke({
    "history": [],
    "question": question
})
print(result)

{
  "category": "Delivery",
  "priority": "Medium",
  "reason": "Package arrival later than expected",
  "sentiment": "Negative",
  "recommended_action": "Apologize for the delay, investigate the cause of the late delivery, update the customer on the current status of the package, and offer a goodwill gesture such as a discount or future shipping credit if applicable."
}
